In [1]:
import pandas as pd
import numpy as np
from joblib import Parallel, delayed
import requests
import pickle
import random
from scipy.interpolate import griddata
from scipy.optimize import fsolve
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
# pip install --upgrade pypa

In [2]:
from age_pension import calculate_pension_income
from CHSP_GRandF import GRandF_CHSP
from HCP_GRandF import GRandF_HCP
from RC_GRandF import GRandF_RC
from comInterp import compress_interp
from FIstatus import I_status, F_status
from DeltaW import Delta_W_status
from updown_prep import derivatives_tp1, direct2EI, Hstate2Cf, reVC, Compare35
from AIP import xiatt, Hstate2IRACF

In [3]:
# This is the case when the total initial wealth = 560000 

def return_calib_var(alpha = 0.5):
    Leasing = 1
    W =  560000 
    H = W * alpha/(alpha+1)
    tau1_arr = np.full(41, 1/6)
    tau2_HCP4 = np.full(41, 3/4)
    tau2_RC = np.full(41, 1/24)
    return H, W/(1+alpha), tau1_arr, tau2_HCP4, tau2_RC, Leasing


In [4]:
ratios = np.linspace(0.05, 0.95, 25)
alphas = ratios / (1 - ratios)  # Solve alpha / (1 + alpha) = ratio for alpha

C_low = 10000
C_RACF = 17961
rho = 2
gamma = 5
theta = (1-gamma)/(1-rho)
b = 2
beta = 0.96
# W_max = 2000000
# _,Wd1pa,_,_,_,_= return_calib_var(0)
# lambda1 = Wd1pa/W_max 

In [5]:
with open('Dict_35', 'rb') as file:
    Dict_35 = pickle.load(file)

with open('National_Transit.pkl', 'rb') as file:
    National_Transit = pickle.load(file)

with open('ME_grid_H.pkl', 'rb') as file:
    ME_grid_H = pickle.load(file)

with open('ME_grid_ir.pkl', 'rb') as file:
    ME_grid_ir = pickle.load(file)

with open('ME_grid_cpi.pkl', 'rb') as file:
    ME_grid_cpi = pickle.load(file)

ME_grid_cpi[0] = [1]
ME_grid_H[0] = [1]

In [6]:
def wealth_law(EI_t, W_t, C_t, alpha, t, Hadded, Hstate):
    H_ini, W_ini, tau1_arr, tau2_HCP4_arr, tau2_RC_arr, Leasing = return_calib_var(alpha)
    r_t = ME_grid_ir[t][EI_t]/100
    cpi_t = ME_grid_cpi[t][EI_t]
    H_t = H_ini * ME_grid_H[t][EI_t] / cpi_t
    I_t = I_status(W_t, H_t, Hstate)
    F_t = F_status(W_t, H_t, Hstate)
    W_tp1 = (W_t - C_t + I_t - F_t)*(1 + r_t) + H_t*Hadded
    return W_tp1

In [7]:
def up_down_situation(EI_tm1, W_t, C_t, alpha, Hstate, t, direction, Dict_interp_tp1):
    EI_t = direct2EI(EI_tm1, direction)
    C_f_t = Hstate2Cf(Hstate)
    H_ini, W_ini, tau1_arr, tau2_HCP4_arr, tau2_RC_arr, Leasing = return_calib_var(alpha)
    r_t = ME_grid_ir[t][EI_t]/100
    cpi_t = ME_grid_cpi[t][EI_t]
    H_t = H_ini * ME_grid_H[t][EI_t] / cpi_t
    W_tp1 = max(wealth_law(EI_t, W_t, C_t, alpha, t, 0, Hstate), C_f_t)
    W_tp1_selling = max(wealth_law(EI_t, W_t, C_t, alpha, t, 1, Hstate), 1)
    if t == 40:
        expectation_1 = b ** gamma * W_tp1_selling ** (1 - gamma)
        expectation_2 = (1 + r_t) * b ** gamma / (1 - beta)* (1/xiatt(t,alpha,0))**(1-rho) * W_tp1_selling ** (-gamma)
    if t < 40:
        pi_t = National_Transit[t]
        V1, C1 = reVC(W_tp1, EI_t, 1, Dict_interp_tp1)
        V2, C2 = reVC(W_tp1, EI_t, 2, Dict_interp_tp1)
        mu1 = 1 + derivatives_tp1(W_tp1, H_t, 1)
        mu2 = 1 + derivatives_tp1(W_tp1, H_t, 2)
        V3, C3 = reVC(W_tp1 - Delta_W_status(W_tp1, H_t, 3, tau1_arr[t], tau2_HCP4_arr[t], tau2_RC_arr[t]), EI_t, 3, Dict_interp_tp1)
        V5, C5 = reVC(W_tp1 - Delta_W_status(W_tp1, H_t, 5, tau1_arr[t], tau2_HCP4_arr[t], tau2_RC_arr[t]), EI_t, 5, Dict_interp_tp1)
        V3s, C3s, mu3s, I_RACF_tp1 = Compare35(W_tp1, H_t, alpha, t, V3, C3, V5, C5)
        expectation_1 = pi_t[Hstate-1,0] * V1 **(1-gamma)\
                      + pi_t[Hstate-1,1] * V2 **(1-gamma)\
                      + pi_t[Hstate-1,2] * V3s**(1-gamma)\
                      + pi_t[Hstate-1,3] * b ** gamma * W_tp1_selling ** (1 - gamma)
        
        expectation_2 = pi_t[Hstate-1,0]*(1 + r_t)* V1 **(rho-gamma)* C1 **(-rho)* mu1*(xiatt(t+1,alpha,0)/xiatt(t,alpha,0))**(1-rho)\
                      + pi_t[Hstate-1,1]*(1 + r_t)* V2 **(rho-gamma)* C2 **(-rho)* mu2*(xiatt(t+1,alpha,0)/xiatt(t,alpha,0))**(1-rho)\
                      + pi_t[Hstate-1,2]*(1 + r_t)* V3s**(rho-gamma)* C3s**(-rho)* mu3s*(xiatt(t+1,alpha,I_RACF_tp1)/xiatt(t,alpha,0))**(1-rho)\
                      + pi_t[Hstate-1,3]*(1 + r_t)* b ** gamma/(1-beta) * (1/xiatt(t,alpha,0))**(1-rho) * W_tp1_selling ** (-gamma)
        
    return expectation_1, expectation_2

In [8]:
def C_selection_t(EI_tm1, W_t, C_t, alpha, Hstate, t, Dict_interp_tp1):
    expectation_1_up, expectation_2_up = up_down_situation(EI_tm1, W_t, C_t, alpha, Hstate, t,'up', Dict_interp_tp1) # goes up 
    expectation_1_down, expectation_2_down = up_down_situation(EI_tm1, W_t, C_t, alpha, Hstate, t,'down', Dict_interp_tp1) #  goes down 
    expectation_1 = (expectation_1_up + expectation_1_down)/2
    expectation_2 = (expectation_2_up + expectation_2_down)/2
    C_t_star = (beta*(expectation_1**(1/theta - 1))*expectation_2) ** (-1/rho)
    return (C_t_star - C_t)

In [9]:
def find_C_ub_direct(EI_tm1, direction, W_t, alpha, Hstate, t):
    EI_t = direct2EI(EI_tm1, direction)
    H_ini, W_ini, tau1_arr, tau2_HCP4_arr, tau2_RC_arr, Leasing = return_calib_var(alpha)
    r_t = ME_grid_ir[t][EI_t]/100
    cpi_t = ME_grid_cpi[t][EI_t]
    H_t = H_ini * ME_grid_H[t][EI_t] / cpi_t
    I_t = I_status(W_t, H_t, Hstate)
    F_t = F_status(W_t, H_t, Hstate)
    if Hstate <= 2: 
        W_tp1_lowest_3 = Delta_W_status(0, H_t, 3, tau1_arr[t], tau2_HCP4_arr[t], tau2_RC_arr[t])
        W_tp1_lowest_5 = Delta_W_status(0, H_t, 5, tau1_arr[t], tau2_HCP4_arr[t], tau2_RC_arr[t])
        W_tp1_lowest = min(W_tp1_lowest_3, W_tp1_lowest_5) + 10
    if Hstate ==3: 
        W_tp1_lowest = C_low + 10
    if Hstate ==5: 
        W_tp1_lowest = C_RACF + 10
    return W_t + I_t - F_t - W_tp1_lowest/(1 + r_t)

def find_C_ub(EI_tm1, W_t, alpha, Hstate, t):
    c_ub_down= find_C_ub_direct(EI_tm1, 'down', W_t, alpha, Hstate, t)
    c_ub_up  = find_C_ub_direct(EI_tm1, 'up',   W_t, alpha, Hstate, t)
    c_ub = min(c_ub_up, c_ub_down)
    return c_ub

In [10]:
# def find_C_t(EI_tm1, W_t, alpha, Hstate, t, Dict_interp_tp1):
#     if Hstate == 5:
#         C_f_t = C_RACF
#     if Hstate != 5:
#         C_f_t = C_low
#     C_t_solution, _, ier, _ = fsolve(lambda C_t: C_selection_t(EI_tm1, W_t, C_t, alpha, Hstate, t, Dict_interp_tp1), C_f_t, full_output=True)
#     return min(max(C_t_solution[0], C_f_t), max(find_C_ub(EI_tm1, W_t, alpha, Hstate, t), C_f_t))

# # An individual can at most consume the total money at the beginning of each period

In [11]:
def find_C_t(EI_tm1, W_t, alpha, Hstate, t, Dict_interp_tp1):
    C_f_t = C_RACF if Hstate == 5 else C_low
    lower_bound = C_f_t
    upper_bound = max(min(find_C_ub(EI_tm1, W_t, alpha, Hstate, t), W_t), C_f_t+10)
    def function_to_minimize(C_t, EI_tm1, W_t, alpha, Hstate, t, Dict_interp_tp1):
        return abs(C_selection_t(EI_tm1, W_t, C_t, alpha, Hstate, t, Dict_interp_tp1))
    result = minimize_scalar(function_to_minimize, args=(EI_tm1, W_t, alpha, Hstate, t, Dict_interp_tp1), bounds=(lower_bound, upper_bound), method='bounded')
    return result.x
# An individual can at most consume the total money at the beginning of each period

In [12]:
def V_t(EI_tm1, W_t, alpha, Hstate, t, Dict_interp_tp1):
    optimal_C_t = find_C_t(EI_tm1, W_t, alpha, Hstate, t, Dict_interp_tp1)
    expectation_1_up, expectation_2_up = up_down_situation(EI_tm1, W_t, optimal_C_t, alpha, Hstate, t, 'up', Dict_interp_tp1) # goes up 
    expectation_1_down, expectation_2_down = up_down_situation(EI_tm1, W_t, optimal_C_t, alpha, Hstate,  t, 'down', Dict_interp_tp1) #  goes down 
    expectation_1 = (expectation_1_up + expectation_1_down)/2
    V = ((1-beta) * (xiatt(t,alpha,0) * optimal_C_t)**(1-rho) + beta*expectation_1**(1/theta))**(1/(1-rho))
    return V, optimal_C_t


In [13]:
# Generate the grids for downsizing options, varying on the intial relocation of liquid and illiquid wealth 
def gen_grid(alpha, numpoints, Hstate):
    Dict_grid = {}
    C_upper = 60000
    C_lower = 10000
    H_ini, W_ini, tau1_arr, tau2_HCP4_arr, tau2_RC_arr, Leasing = return_calib_var(alpha)
    W_lower_previous = 30000
    W_upper_previous = 1200000
    for t in range(1,41):
        grid = np.linspace(W_lower_previous, W_upper_previous, numpoints)
        Dict_grid[t]= grid
        W_upper = wealth_law(int(t/2), W_upper_previous, C_lower, alpha, t, 0, 1)
        W_upper_previous = W_upper
        W_lower = wealth_law(int(t/2), W_lower_previous, C_upper, alpha, t, 0, 2)
        cpi_t = ME_grid_cpi[t][0]
        H_t = H_ini * ME_grid_H[t][0]/cpi_t
        if Hstate <= 2:
            W_lowest_RACF = C_RACF + Delta_W_status(W_lower, H_t, 5, tau1_arr[t], tau2_HCP4_arr[t], tau2_RC_arr[t])
            W_lowest_HCP4 = C_low  + Delta_W_status(W_lower, H_t, 3, tau1_arr[t], tau2_HCP4_arr[t], tau2_RC_arr[t])
            W_lowest = min(W_lowest_RACF, W_lowest_HCP4) + 10
        W_lower_previous = max(W_lower, W_lowest)
    return Dict_grid


In [14]:
def add_interpolated_CV(t, alpha, Dict_grids, V1_t, V2_t):
    Dict_interpolated_V1_= {}
    Dict_interpolated_V2_= {}
    
    Dict_interpolated_C1_= {}
    Dict_interpolated_C2_= {}

    for EI_t in range(t):
        V1_array = np.array([V1_t[(EI_t, W)][0] for W in Dict_grids[1][t]])
        C1_array = np.array([V1_t[(EI_t, W)][1] for W in Dict_grids[1][t]])

        V1_interpolated = interp1d(Dict_grids[1][t], V1_array, kind='linear', fill_value= "extrapolate")
        C1_interpolated = interp1d(Dict_grids[1][t], C1_array, kind='linear', fill_value= "extrapolate")
    
        V2_array = np.array([V2_t[(EI_t, W)][0] for W in Dict_grids[2][t]])
        C2_array = np.array([V2_t[(EI_t, W)][1] for W in Dict_grids[2][t]])

        V2_interpolated = interp1d(Dict_grids[2][t], V2_array, kind='linear', fill_value= "extrapolate")
        C2_interpolated = interp1d(Dict_grids[2][t], C2_array, kind='linear', fill_value= "extrapolate")

        Dict_interpolated_V1_[EI_t] =  V1_interpolated
        Dict_interpolated_V2_[EI_t] =  V2_interpolated

        Dict_interpolated_C1_[EI_t] =  C1_interpolated
        Dict_interpolated_C2_[EI_t] =  C2_interpolated

    return  Dict_interpolated_V1_, Dict_interpolated_V2_, Dict_interpolated_C1_, Dict_interpolated_C2_

In [15]:
def calculate_Dicts(alpha):
    numpoints = 25
    H_ini, W_ini, tau1_arr, tau2_HCP4_arr, tau2_RC_arr, Leasing = return_calib_var(alpha)
    estimate_prop = H_ini/1200000
    closest_index = np.argmin(np.abs(ratios - estimate_prop))

    dict_grids = {}
    for Hstate in [1,2]:
        dict_grids[Hstate] =  gen_grid(alpha, numpoints, Hstate)
    Dict_interpolated_V3 = Dict_35[closest_index]['WV3']
    Dict_interpolated_V5 = Dict_35[closest_index]['WV5']
    Dict_interpolated_C3 = Dict_35[closest_index]['WC3']
    Dict_interpolated_C5 = Dict_35[closest_index]['WC5']
    Dict_interpolated_V1 = {}
    Dict_interpolated_V2 = {}
    Dict_interpolated_C1 = {}
    Dict_interpolated_C2 = {}
    V1_= {}
    V2_= {}
    for t in range(40, 0, -1):
        EI_tm1_range = range(t)
        V1_current = {}
        V2_current = {}
        if t == 40:
            Dict_interpolated_V1_tp1 = {}
            Dict_interpolated_C1_tp1 = {}
            Dict_interpolated_V2_tp1 = {}
            Dict_interpolated_C2_tp1 = {}
            Dict_interpolated_V3_tp1 = {}
            Dict_interpolated_C3_tp1 = {}
            Dict_interpolated_V5_tp1 = {}
            Dict_interpolated_C5_tp1 = {}
        if t < 40:
            Dict_interpolated_V1_tp1 = Dict_interpolated_V1[t+1]
            Dict_interpolated_C1_tp1 = Dict_interpolated_C1[t+1]
            Dict_interpolated_V2_tp1 = Dict_interpolated_V2[t+1]
            Dict_interpolated_C2_tp1 = Dict_interpolated_C2[t+1]
            Dict_interpolated_V3_tp1 = Dict_interpolated_V3[t+1]
            Dict_interpolated_C3_tp1 = Dict_interpolated_C3[t+1]
            Dict_interpolated_V5_tp1 = Dict_interpolated_V5[t+1]
            Dict_interpolated_C5_tp1 = Dict_interpolated_C5[t+1]
        Dict_interp_tp1 = compress_interp(Dict_interpolated_V1_tp1, Dict_interpolated_C1_tp1, Dict_interpolated_V2_tp1, Dict_interpolated_C2_tp1, Dict_interpolated_V3_tp1, Dict_interpolated_C3_tp1, Dict_interpolated_V5_tp1, Dict_interpolated_C5_tp1)
        for EI_tm1 in EI_tm1_range:
            for W_t in dict_grids[1][t]:
                # Call the function and store the result
                key = (EI_tm1, W_t)
                V1_current[key] = V_t(EI_tm1, W_t, alpha, 1, t, Dict_interp_tp1)
            for W_t in dict_grids[2][t]:
                # Call the function and store the result
                key = (EI_tm1, W_t)
                V2_current[key] = V_t(EI_tm1, W_t, alpha, 2, t, Dict_interp_tp1)
        V1_[t] = V1_current
        V2_[t] = V2_current
        results = add_interpolated_CV(t, alpha, dict_grids, V1_current, V2_current)
        Dict_interpolated_V1[t] = results[0]
        Dict_interpolated_V2[t] = results[1]
        Dict_interpolated_C1[t] = results[2]
        Dict_interpolated_C2[t] = results[3]
    return V1_,V2_, Dict_interpolated_V1, Dict_interpolated_V2, Dict_interpolated_C1, Dict_interpolated_C2


In [16]:
results = Parallel(n_jobs=25)(delayed(calculate_Dicts)(alphas[i]) for i in range(25))
Dict_12 = {}
for i in range(25):
    Dict_12[i] = {}
    Dict_12[i]['V1']  = [res[0] for res in results][i]
    Dict_12[i]['V2']  = [res[1] for res in results][i]
    Dict_12[i]['WV1'] = [res[2] for res in results][i]
    Dict_12[i]['WV2'] = [res[3] for res in results][i]
    Dict_12[i]['WC1'] = [res[4] for res in results][i]
    Dict_12[i]['WC2'] = [res[5] for res in results][i]
with open('Dict_12', 'wb') as file:
    pickle.dump(Dict_12, file)

/home/z5234922/.local/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:700: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
